# Step 3A — Build Targets (ALSFRS-R slope @ 3m / 6m) — leakage-free

## Objective
Build binary targets for **rapid vs slow progression**, using the ALSFRS-R *slope* between:
- **3 months**: 90 ± 7 days  
- **6 months**: 180 ± 7 days  

The baseline (t0) is the **first visit with a valid ALSFRS-R score** per patient.

## Inputs
- `01_data/raw/PROACT_ALSFRS.csv`
- `01_data/interim/baseline_table_ALSFRS_R.csv`  *(t0 already defined per patient)*

## Outputs
- `01_data/interim/baseline_with_targets_step3_nolabelsv2.csv`  
  Contains (per patient): `t0`, 3m/6m slopes (when eligible) and slope quality metrics.

## Parameters (fixed in this notebook)
- temporal tolerance: **±7 days**
- horizons: **90** and **180** days
- "rapid" rule: **top 30% worst slopes** *(applied later during training, fold-wise)*

<div style="padding:10px;border-left:6px solid #FF5F5D;">
<b>Leakage-free note:</b> this notebook only computes slopes/targets using post-baseline measurements; the train/test split by patient is performed downstream (GroupKFold).
</div>


In [1]:
import os
import numpy as np
import pandas as pd

RAW = os.path.join("..", "01_data", "raw")
INTERIM = os.path.join("..", "01_data", "interim")

os.makedirs(INTERIM, exist_ok=True)

als_path = os.path.join(RAW, "PROACT_ALSFRS.csv")
baseline_path = os.path.join(INTERIM, "baseline_table_ALSFRS_R.csv")

als = pd.read_csv(als_path)
base = pd.read_csv(baseline_path)

print("ALS:", als.shape)
print("BASE:", base.shape)

als.head()


ALS: (73845, 20)
BASE: (5436, 18)


,subject_id,Q1_Speech,Q2_Salivation,Q3_Swallowing,Q4_Handwriting,Q5a_Cutting_without_Gastrostomy,Q5b_Cutting_with_Gastrostomy,Q6_Dressing_and_Hygiene,Q7_Turning_in_Bed,Q8_Walking,Q9_Climbing_Stairs,Q10_Respiratory,ALSFRS_Delta,ALSFRS_Total,ALSFRS_R_Total,R_1_Dyspnea,R_2_Orthopnea,R_3_Respiratory_Insufficiency,Mode_of_Administration,ALSFRS_Responded_By
0,121,4.0,4.0,4.0,3.0,3.0,NaN,4.0,2.0,4.0,4.0,4.0,0.0,36.0,NaN,NaN,NaN,NaN,NaN,NaN
1,121,4.0,4.0,4.0,3.0,3.0,NaN,3.0,3.0,4.0,3.0,4.0,28.0,35.0,NaN,NaN,NaN,NaN,NaN,NaN
2,121,4.0,4.0,4.0,3.0,3.0,NaN,3.0,3.0,4.0,4.0,4.0,56.0,36.0,NaN,NaN,NaN,NaN,NaN,NaN
3,121,4.0,4.0,4.0,3.0,3.0,NaN,3.0,3.0,4.0,3.0,4.0,84.0,35.0,NaN,NaN,NaN,NaN,NaN,NaN
4,121,4.0,4.0,4.0,3.0,3.0,NaN,2.0,4.0,4.0,3.0,4.0,116.0,35.0,NaN,NaN,NaN,NaN,NaN,NaN


## 1) Prepare ALSFRS-R measurements
Here we clean and sort the ALSFRS-R measurements per patient, ensuring:
- correct numeric types (delta in days, total score)
- removal/handling of invalid records
- foundation for finding measurements near **t0+90** and **t0+180** (with tolerance)


In [ ]:
als_r = als[["subject_id", "ALSFRS_Delta", "ALSFRS_R_Total"]].copy()

als_r["ALSFRS_Delta"] = pd.to_numeric(als_r["ALSFRS_Delta"], errors="coerce")
als_r["ALSFRS_R_Total"] = pd.to_numeric(als_r["ALSFRS_R_Total"], errors="coerce")

als_r = als_r.dropna(subset=["ALSFRS_Delta", "ALSFRS_R_Total"])
als_r = als_r.sort_values(["subject_id", "ALSFRS_Delta"])

# dict subject -> df for fast lookup
grouped = {int(sid): df for sid, df in als_r.groupby("subject_id")}

print("Subjects ALSFRS-R:", als_r["subject_id"].nunique())


Subjects ALSFRS-R: 5436


## 2) Compute slope per patient (linear regression) with window and tolerance

For each patient and horizon (H):
1) fix the baseline **t0**
2) find measurements in the interval **[t0, t0+H]** and confirm a data point exists near **t0+H** (±7 days)
3) estimate the slope via linear regression (points/day and points/30 days)

In addition to the slope, we store quality indicators:
- number of data points used
- effective temporal span
- (if applicable) R² / errors


In [ ]:
def slope_for_subject(sid:int, t0:float, H:int, tol:int=7):
    """
    Slope in points/day and points/30d, using linear regression on [t0, t0+H+tol].
    Requires at least 1 measurement near the target: [t0+H-tol, t0+H+tol].
    """
    df = grouped.get(int(sid))
    if df is None:
        return (np.nan, np.nan, 0, False)

    start = t0
    end = t0 + H + tol
    end_lo = t0 + H - tol
    end_hi = t0 + H + tol

    w = df[(df["ALSFRS_Delta"] >= start) & (df["ALSFRS_Delta"] <= end)].copy()
    if w.empty:
        return (np.nan, np.nan, 0, False)

    has_endband = ((w["ALSFRS_Delta"] >= end_lo) & (w["ALSFRS_Delta"] <= end_hi)).any()
    if not has_endband:
        return (np.nan, np.nan, int(w.shape[0]), False)

    if w["ALSFRS_Delta"].nunique() < 2:
        return (np.nan, np.nan, int(w.shape[0]), True)

    x = (w["ALSFRS_Delta"] - t0).to_numpy(dtype=float)  # days since t0
    y = w["ALSFRS_R_Total"].to_numpy(dtype=float)

    slope, intercept = np.polyfit(x, y, 1)  # slope in points/day
    return (float(slope), float(slope*30.0), int(w.shape[0]), True)


## 3) Generate targets (3m and 6m) and merge with baseline

In this step:
- we compute slopes for **90±7** and **180±7**
- each patient may be eligible for 3m, 6m, both, or neither (depending on coverage)
- we merge with the baseline table (1 row per patient)

The result is the "master table" for the subsequent steps (slope EDA and model training).


In [ ]:
base2 = base[["subject_id", "t0_delta_days"]].copy()
base2["t0_delta_days"] = pd.to_numeric(base2["t0_delta_days"], errors="coerce")
base2 = base2.dropna(subset=["t0_delta_days"])

rows = []
for sid, t0 in zip(base2["subject_id"], base2["t0_delta_days"]):
    sid = int(sid)
    t0 = float(t0)

    row = {"subject_id": sid}

    for H in [90, 180]:
        s_day, s_30d, npts, ok_end = slope_for_subject(sid, t0, H, tol=7)
        row[f"slope_{H}d_per_day"] = s_day
        row[f"slope_{H}d_per_30d"] = s_30d
        row[f"npoints_{H}d_window"] = npts
        row[f"has_endband_{H}d"] = ok_end

    rows.append(row)

targets = pd.DataFrame(rows)

# flags "preferred >=3 data points"
targets["pref_ok_90d_min3"]  = targets["npoints_90d_window"]  >= 3
targets["pref_ok_180d_min3"] = targets["npoints_180d_window"] >= 3

baseline_with_targets = base.merge(targets, on="subject_id", how="left")

baseline_with_targets[[
    "slope_90d_per_30d","slope_180d_per_30d",
    "npoints_90d_window","npoints_180d_window",
    "has_endband_90d","has_endband_180d"
]].head()


,slope_90d_per_30d,slope_180d_per_30d,npoints_90d_window,npoints_180d_window,has_endband_90d,has_endband_180d
0,-2.564168,-1.524227,3,4,True,True
1,NaN,NaN,4,7,False,False
2,0.991344,-0.117096,4,7,True,True
3,-0.666667,NaN,2,2,True,False
4,NaN,NaN,2,3,False,False


## 4) Save intermediate dataset (targets + slopes)

We save only the information needed for:
- EDA in Step 3B
- building the final dataset in Steps 4/5

Note: **we do not save global labels** here (e.g., "rapid" by global percentile),
because the binary label should be defined in a controlled manner during training (ideally fold-wise).


In [ ]:
out_path = os.path.join(INTERIM, "baseline_with_targets_step3_nolabelsv2.csv")
baseline_with_targets.to_csv(out_path, index=False)

print("Saved:", out_path)
print("Total baseline N:", len(baseline_with_targets))
print("Eligible 3m:", baseline_with_targets["slope_90d_per_30d"].notna().sum())
print("Eligible 6m:", baseline_with_targets["slope_180d_per_30d"].notna().sum())
print("Eligible both:",
      (baseline_with_targets["slope_90d_per_30d"].notna() &
       baseline_with_targets["slope_180d_per_30d"].notna()).sum())


Guardado: ..\01_data\interim\baseline_with_targets_step3_nolabelsv2.csv
N total baseline: 5436
Eligible 3m: 2407
Eligible 6m: 1392
Eligible both: 720
